In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, and data handling

# Data Loading and Inspecting

In [2]:
data = pd.read_csv("data/ecommerceDataset.csv", names=["Category","Text"])

In [3]:
data = data[["Text","Category"]]

In [ ]:
data.shape
data.head()

(50425, 2)

In [5]:
data.iloc[0,:]["Text"]

'Paper Plane Design Framed Wall Hanging Motivational Office Decor Art Prints (8.7 X 8.7 inch) - Set of 4 Painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch. This painting is ready to hang, you would be proud to possess this unique painting that is a niche apart. We use only the most modern and efficient printing technology on our prints, with only the and inks and precision epson, roland and hp printers. This innovative hd printing technique results in durable and spectacular looking prints of the highest that last a lifetime. We print solely with top-notch 100% inks, to achieve brilliant and true colours. Due to their high level of uv resistance, our prints retain their beautiful colours for many years. Add colour and style to your living space with this digitally printed painting. Some are for pleasure and some for eternal blis

In [6]:
data["Category"].value_counts()

Category
Household                 19313
Books                     11820
Electronics               10621
Clothing & Accessories     8671
Name: count, dtype: int64

# Textual Processing

In [7]:
import re

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    
    text = re.sub(r"\'s\b", " is", text)     # it's -> it is, John's -> John is
    text = re.sub(r"n\'t\b", " not", text)   # wasn't -> was not
    text = re.sub(r"\'re\b", " are", text)   # they're -> they are
    text = re.sub(r"\'ll\b", " will", text)  # i'll -> i will
    text = re.sub(r"\'m\b", " am", text)     # i'm -> i am
    text = re.sub(r"\'ve\b", " have", text)  # we've -> we have
    text = re.sub(r"\'d\b", " would", text)  # he'd -> he would
    
    # 2. Handle Smart Quotes and Backticks
    # This converts “ ” and ‘ ’ to standard ' and "
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[‘’]', "'", text)
    text = text.replace('`', "'")
    
    # 3. Remove Quotes (or replace with space to avoid merging words)
    # We use a space to ensure "word"word" becomes "word word"
    text = re.sub(r'[\"\']', ' ', text)
    
    # 4. Standard Cleanup (URLs, Handles)
    text = re.sub(r'http\S+|www\S+|@\w+', '', text)
    
    # 5. Remove non-alphanumeric (keeping basic punctuation if you want)
    # If you want to strip ALL punctuation, use [^a-z0-s\s]
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 6. Final whitespace cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r"\b x\b", "", text)
    return text


In [8]:
data["Text_normalised"] = data["Text"].apply(normalize_text)

In [9]:
from random import randint
# row = 5000
row = randint(0,len(data))
print(f"Original Text Row {row}\n",data.iloc[row,0],"\n")
print(f"Normalized Text Row {row}\n",data.iloc[row,2])

Original Text Row 12739
 Rrimin Stainless Steel Domestic Sewing Machine Foot Feet Snap on for Brother Singer Set-Set of 32 Fits Most Sewing Machine Brands Such As Singer, Brother, Babylock, Euro-Pro, Janome, Kenmore, White, Juki, Home, Necchi, Elna, Husqvarna Viking, Toyota and Other Low-Shank Domestic Sewing Machines Features: Quantity:1 Set (32Pcs) Case Size:235 X 200 X 27 Mm Color:As The Picture Shows Weight:338 G Package Includes 32 Pieces Of Feet, The Listing As Below: A: 1. Braiding Foot; 2. Fringe Foot; 3. 5 Hole Cording Foot; 4. 7 Hole Cording Foot; 5. Edge Joining Foot; 6.1/4" Quilting Foot; 7.Straight Stitch Foot B: 1. Open Toe Embroidery Foot; 2. Round Bead Foot; 3. Zig Zag Foot; 4. Cording Foot; 5. 1/4" Quilting Foot; 6.Pintuck Foot 9 Grooves; 7.Overcast Foot C: 1. Stitch Guide Foot; 2. Pintuck Foot 5 Grooves; 3. Satin Stitch Foot; 4. Zig Zag Foot, Jaguar; 5. Double Welting Foot; 6. Invisible Zipper Foot; 7. Roller Foot D: 1. Blind Stitch Foot; 2. Applique Foot; 3. Open Toe

# Leakage-Safe Text Classification Pipeline

**Important change:** we split the raw text into train/test **before fitting TF-IDF**. The TF-IDF vocabulary and IDF statistics are learned from the training data only.

We will compare:
- Logistic Regression
- Linear SVM (`LinearSVC`)
- Random Forest
- XGBoost

Model selection uses **StratifiedKFold cross-validation on the training set**, using macro-F1 so the weaker/minority classes are not hidden by the larger `Household` class.

In [10]:
# Core imports
import re
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from xgboost import XGBClassifier

Matplotlib is building the font cache; this may take a moment.


## 1. Text cleaning — with a number-preserving option

The original cleaner removed every digit. That can destroy useful product signals such as `4K`, `128GB`, `15.3W`, `65-inch`, etc.

We therefore keep **two cleaning modes**:
1. `keep_numbers=False` — the original-style alphabetic-only representation.
2. `keep_numbers=True` — preserves alphanumeric product specifications.

We will run an ablation later to see whether preserving these signals actually improves performance.

In [11]:
def normalize_text(text, keep_numbers=True):
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Expand common contractions before removing punctuation.
    text = re.sub(r"\'s\b", " is", text)
    text = re.sub(r"n\'t\b", " not", text)
    text = re.sub(r"\'re\b", " are", text)
    text = re.sub(r"\'ll\b", " will", text)
    text = re.sub(r"\'m\b", " am", text)
    text = re.sub(r"\'ve\b", " have", text)
    text = re.sub(r"\'d\b", " would", text)

    # Normalize smart quotes/backticks.
    text = re.sub(r'[“”]', '"', text)
    text = re.sub(r'[‘’]', "'", text)
    text = text.replace('`', "'")

    # Remove URLs/handles.
    text = re.sub(r'http\S+|www\S+|@\w+', ' ', text)

    if keep_numbers:
        # Keep letters, digits and whitespace. This retains signals such as
        # 128gb, 4k, 15.3w (the decimal point itself is removed).
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
    else:
        text = re.sub(r'[^a-z\s]', ' ', text)

    # Whitespace cleanup.
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [12]:
# Create both representations before splitting.
# This step is deterministic and does not learn anything from the data.
data["Text_alpha_only"] = data["Text"].apply(lambda x: normalize_text(x, keep_numbers=False))
data["Text_with_numbers"] = data["Text"].apply(lambda x: normalize_text(x, keep_numbers=True))

# Quick sanity check for the useful numeric signals.
sample_text = "Official USB-C 15.3W Power Supply, 4K display, 128GB storage"
print("Alpha-only :", normalize_text(sample_text, keep_numbers=False))
print("With nums  :", normalize_text(sample_text, keep_numbers=True))

Alpha-only : official usb c w power supply k display gb storage
With nums  : official usb c 15 3w power supply 4k display 128gb storage


## 2. Split first — before fitting TF-IDF or the classifier

We split the **raw text representations** and labels first. The test set remains completely untouched during model selection.

In [13]:
# Fit the label mapping using the training labels only.
# Label encoding itself is not a feature transformation, but keeping the mapping
# tied to training data makes the train/test workflow explicit and deployment-safe.
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    data["Text_with_numbers"],
    data["Category"],
    test_size=0.20,
    stratify=data["Category"],
    random_state=2306
)

encode = LabelEncoder()
y_train = encode.fit_transform(y_text_train)
y_test = encode.transform(y_text_test)

print("Classes:", encode.classes_)
print("Train size:", len(X_text_train))
print("Test size :", len(X_text_test))

Classes: ['Books' 'Clothing & Accessories' 'Electronics' 'Household']
Train size: 40340
Test size : 10085


## 3. TF-IDF configuration

TF-IDF now lives **inside each scikit-learn Pipeline**. During cross-validation, each fold fits its own vectorizer only on that fold's training portion. This removes the original leakage caused by calling `tfidf.fit_transform()` on the full dataset before the split.

In [14]:
def make_tfidf():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=15000,
        min_df=5,
        max_df=0.7,
        stop_words="english"
    )

## 4. Compare multiple algorithms with StratifiedKFold

We use **macro-F1** as the primary selection metric because the classes are imbalanced. Accuracy is still reported later, but it is not the only number we use to choose the model.

In [15]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=2.0,
        solver="lbfgs",
        random_state=42
    ),
    "Linear SVM": LinearSVC(
        C=1.0,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=1,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        objective="multi:softprob",
        num_class=len(encode.classes_),
        tree_method="hist",
        eval_metric="mlogloss",
        n_jobs=-1,
        random_state=42
    )
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    pipe = Pipeline([
        ("tfidf", make_tfidf()),
        ("model", model)
    ])

    scores = cross_val_score(
        pipe,
        X_text_train,
        y_train,
        cv=skf,
        scoring="f1_macro",
        n_jobs=1
    )

    cv_results.append({
        "Model": name,
        "Mean CV Macro-F1": scores.mean(),
        "Std CV Macro-F1": scores.std(),
        "Fold 1": scores[0],
        "Fold 2": scores[1],
        "Fold 3": scores[2],
        "Fold 4": scores[3],
        "Fold 5": scores[4]
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "Mean CV Macro-F1", ascending=False
).reset_index(drop=True)

cv_results_df

KeyboardInterrupt: 

### Select the best model

The winner is selected from the **mean 5-fold macro-F1 on the training set**. The test set is not used for this decision.

In [ ]:
best_model_name = cv_results_df.loc[0, "Model"]
best_model = models[best_model_name]

print(f"Selected model: {best_model_name}")
print(f"Mean CV Macro-F1: {cv_results_df.loc[0, 'Mean CV Macro-F1']:.4f}")

## 5. Fit the selected model once on the full training set and evaluate on the held-out test set

In [ ]:
final_pipeline = Pipeline([
    ("tfidf", make_tfidf()),
    ("model", best_model)
])

final_pipeline.fit(X_text_train, y_train)
y_pred = final_pipeline.predict(X_text_test)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"Test Accuracy     : {accuracy:.4f}")
print(f"Test Macro-F1     : {macro_f1:.4f}")
print(f"Test Weighted-F1  : {weighted_f1:.4f}")

print("\nClassification Report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=encode.classes_,
    digits=4
))

## 6. Readable confusion matrix

The class labels are explicitly supplied, so the matrix is interpretable instead of showing only encoded values `0, 1, 2, 3`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=encode.classes_,
    xticks_rotation=30,
    cmap="Blues",
    ax=ax
)
ax.set_title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 7. Number-preservation ablation

This directly tests whether retaining product specifications such as `128GB`, `4K`, `15.3W`, and similar tokens helps classification.

To keep the ablation focused, we compare the **same selected algorithm** under both preprocessing choices. Both pipelines still fit TF-IDF only inside the CV folds.

In [ ]:
X_alpha_train, X_alpha_test, _, _ = train_test_split(
    data["Text_alpha_only"],
    data["Category"],
    test_size=0.20,
    stratify=data["Category"],
    random_state=2306
)

# Re-use the same training indices implicitly by using the same random_state/stratification.
# Sanity check that the labels align exactly with the earlier split.
assert X_alpha_train.index.equals(X_text_train.index)
assert X_alpha_test.index.equals(X_text_test.index)

alpha_pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("model", models[best_model_name])
])

numeric_pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("model", models[best_model_name])
])

alpha_scores = cross_val_score(
    alpha_pipe, X_alpha_train, y_train, cv=skf, scoring="f1_macro", n_jobs=1
)
numeric_scores = cross_val_score(
    numeric_pipe, X_text_train, y_train, cv=skf, scoring="f1_macro", n_jobs=1
)

ablation_results = pd.DataFrame({
    "Preprocessing": ["Remove numbers", "Keep alphanumeric signals"],
    "Mean CV Macro-F1": [alpha_scores.mean(), numeric_scores.mean()],
    "Std CV Macro-F1": [alpha_scores.std(), numeric_scores.std()]
})

ablation_results["Delta vs alpha-only"] = (
    ablation_results["Mean CV Macro-F1"] - alpha_scores.mean()
)

ablation_results

### Interpretation

- If **Keep alphanumeric signals** has higher macro-F1, retain the number-preserving cleaner.
- If the scores are essentially equal, prefer the simpler representation unless the per-class results show a meaningful improvement for Electronics.
- If it performs worse, keep the alpha-only representation.

The important point is that this is now an **experiment**, not an assumption.

## 8. Train the final deployment artifacts

For deployment we need three persistent artifacts:
1. `tfidf_vectorizer.joblib` — the vocabulary/IDF learned from training data.
2. `best_model.joblib` — the selected classifier trained on the full training split.
3. `label_encoder.joblib` — maps model class IDs back to category names.

We use the number-preserving representation only if the ablation supports it.

In [ ]:
# Choose preprocessing based on the ablation result.
keep_numbers_for_deployment = numeric_scores.mean() >= alpha_scores.mean()

X_final_train = X_text_train if keep_numbers_for_deployment else X_alpha_train
X_final_test = X_text_test if keep_numbers_for_deployment else X_alpha_test

# Create fresh instances so the deployment artifacts are clean and fitted once.
selected_model = models[best_model_name]
final_vectorizer = make_tfidf()
X_final_train_tfidf = final_vectorizer.fit_transform(X_final_train)

selected_model.fit(X_final_train_tfidf, y_train)

# Final held-out test evaluation for the representation chosen by the ablation.
X_final_test_tfidf = final_vectorizer.transform(X_final_test)
final_test_pred = selected_model.predict(X_final_test_tfidf)

print("Deployment preprocessing:", "keep numbers" if keep_numbers_for_deployment else "alpha-only")
print(f"Final test accuracy: {accuracy_score(y_test, final_test_pred):.4f}")
print(f"Final test macro-F1: {f1_score(y_test, final_test_pred, average='macro'):.4f}")

In [ ]:
# Save exactly the three artifacts needed by the application.
joblib.dump(final_vectorizer, "tfidf_vectorizer.joblib")
joblib.dump(selected_model, "best_model.joblib")
joblib.dump(encode, "label_encoder.joblib")

print("Saved:")
print(" - tfidf_vectorizer.joblib")
print(" - best_model.joblib")
print(" - label_encoder.joblib")

## 9. Deployment-style prediction

The application must perform the **same cleaning → TF-IDF transform → prediction → label decoding** sequence used during training.

In [ ]:
# Load the exact artifacts that will be used by the deployed app.
loaded_vectorizer = joblib.load("tfidf_vectorizer.joblib")
loaded_model = joblib.load("best_model.joblib")
loaded_encoder = joblib.load("label_encoder.joblib")

text = """The Official USB type C 15.3W Power Supply For Raspberry Pi 4 is the Latest officially launched power supply for all new Raspberry Pi 4 Model.
The adapter comes with 1.5m 18 AWG captive cable with a USB-C output connector.
Original official Raspberry Pi adapter with a Raspberry PI LOGO engraved on the top.
USB-C connection goes straight into your Raspberry Pi.
96-230Vac operating input range. 50,000 hours MTBF."""

cleaned = normalize_text(text, keep_numbers=keep_numbers_for_deployment)
features = loaded_vectorizer.transform([cleaned])
prediction = loaded_model.predict(features)[0]

print("Predicted category:", loaded_encoder.inverse_transform([prediction])[0])

## Final ML lifecycle

**Raw data → deterministic cleaning → train/test split → TF-IDF fitted on train only → StratifiedKFold model comparison → select best model → held-out test evaluation → number-preservation ablation → refit selected representation/model on training data → save vectorizer + model + label encoder → deployment prediction.**

This structure avoids the original TF-IDF leakage, gives a defensible model comparison, makes the confusion matrix readable, tests the numeric-token hypothesis, and produces the artifacts required by the deployed application.